# Shared HMM helpers

Directed topology is checked by reachability through the complete raw road graph, using strongly connected components. A legal multi-edge route is not called disconnected merely because endpoints do not touch.

The transition distance is a **lower-bound surrogate**, not a shortest-road-path distance: remaining previous segment + straight bridge between endpoints + distance along the next segment. Forward motion on the same arc uses the progress distance. Penalize mismatch against observed displacement, plus a finite penalty for directed-unreachable transitions. Finite penalties allow a decoded output even for inconsistent candidate sets; such jumps remain visible in evaluation.

Larger beta strengthens transition costs. Normalize each transition row. Emission weight scales log probabilities. Reset after missing observations, nonpositive time gaps or gaps over 30 seconds. Viterbi finds the joint MAP sequence; forward/backward produces separate marginal posterior probabilities for honest decoder log loss. The marginal argmax need not equal the Viterbi state.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import shapely
from pyproj import Transformer
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components, breadth_first_order
from scipy.special import logsumexp
from time import perf_counter

In [ ]:
def load_network(trip_id, partition="train", data_root="Student folder"):
    folder = Path(data_root) / partition / trip_id
    nodes = pd.read_csv(folder / f"{trip_id}.nodes", sep=r"\s+", header=None).to_numpy()
    arcs = pd.read_csv(folder / f"{trip_id}.arcs", sep=r"\s+", header=None).to_numpy()
    track = pd.read_csv(folder / f"{trip_id}.track", sep=r"\s+", header=None, names=["lon", "lat", "time"])
    zone = int((track["lon"].mean() + 180) // 6) + 1
    epsg = (32600 if track["lat"].mean() >= 0 else 32700) + zone
    projection = Transformer.from_crs(4326, epsg, always_xy=True)
    xy = np.column_stack(projection.transform(nodes[:, 0], nodes[:, 1]))
    gps_xy = np.column_stack(projection.transform(track["lon"], track["lat"]))
    graph = csr_matrix((np.ones(len(arcs)), (arcs[:, 0], arcs[:, 1])), shape=(len(nodes), len(nodes)))
    count, component = connected_components(graph, directed=True, connection="strong")
    a, b = component[arcs[:, 0]], component[arcs[:, 1]]
    keep = a != b
    condensation = csr_matrix((np.ones(keep.sum()), (a[keep], b[keep])), shape=(count, count))
    return {"arcs": arcs, "xy": xy, "gps_xy": gps_xy, "time": track["time"].to_numpy(),
            "component": component, "condensation": condensation, "reachable_cache": {}}


def component_reachable(network, a, b):
    if a == b:
        return True
    cache = network["reachable_cache"]
    if a not in cache:
        cache[a] = set(breadth_first_order(network["condensation"], a, directed=True, return_predecessors=False))
    return b in cache[a]


def transition_is_reachable(network, previous_segment_id, next_segment_id):
    if previous_segment_id < 0 or next_segment_id < 0:
        return False
    if previous_segment_id == next_segment_id:
        return True
    arcs = network["arcs"]
    end, start = arcs[previous_segment_id, 1], arcs[next_segment_id, 0]
    components = network["component"]
    return component_reachable(network, int(components[end]), int(components[start]))

In [ ]:
def prepare_trip(scores, point_metadata, network, gap_reset_s=30):
    metadata = point_metadata.sort_values("point_id")
    group_to_point = metadata.set_index("group_id")["point_id"]
    scores = scores.sort_values(["group_id", "segment_id"]).copy()
    scores["point_id"] = scores["group_id"].map(group_to_point)
    assert scores["point_id"].notna().all()
    assert not scores.duplicated(["group_id", "segment_id"]).any()
    np.testing.assert_allclose(scores.groupby("group_id")["score"].sum(), 1, atol=1e-5)
    assert np.isfinite(scores["score"]).all() and scores["score"].between(0, 1).all()
    arcs, xy, gps_xy = network["arcs"], network["xy"], network["gps_xy"]
    length = np.linalg.norm(xy[arcs[:, 1]] - xy[arcs[:, 0]], axis=1)
    frames, costs, previous = [], [], None
    for group_id, frame in scores.groupby("group_id", sort=False):
        p = int(frame["point_id"].iloc[0])
        ids = frame["segment_id"].to_numpy(dtype=int)
        segments = shapely.linestrings(xy[arcs[ids]])
        along = shapely.line_locate_point(segments, shapely.Point(gps_xy[p]))
        current = {"point_id": p, "group_id": int(group_id), "ids": ids, "along": along,
                   "log_emission": np.log(np.clip(frame["score"].to_numpy(), 1e-15, 1))}
        cost = None
        if previous is not None:
            last_p, last_ids = previous["point_id"], previous["ids"]
            dt = network["time"][p] - network["time"][last_p]
            if p == last_p + 1 and 0 < dt <= gap_reset_s:
                last_end = xy[arcs[last_ids, 1]]
                current_start = xy[arcs[ids, 0]]
                bridge = np.linalg.norm(last_end[:, None, :] - current_start[None, :, :], axis=2)
                travel = length[last_ids, None] - previous["along"][:, None] + bridge + along[None, :]
                same = last_ids[:, None] == ids[None, :]
                progress = along[None, :] - previous["along"][:, None]
                travel = np.where(same & (progress >= 0), progress, travel)
                observed = np.linalg.norm(gps_xy[p] - gps_xy[last_p])
                cost = np.abs(travel - observed) / max(20.0, observed)
                reachable = np.empty(cost.shape, dtype=bool)
                components = network["component"]
                source = components[arcs[last_ids, 1]]
                target = components[arcs[ids, 0]]
                for a in np.unique(source):
                    for b in np.unique(target):
                        reachable[np.ix_(source == a, target == b)] = component_reachable(network, int(a), int(b))
                reachable |= same
                cost += (~reachable) * 20.0
                cost = np.minimum(cost, 100).astype("float32")
        frames.append(current)
        costs.append(cost)
        previous = current
    return frames, costs


def transition_logprob(cost, beta):
    values = -beta * cost
    return values - logsumexp(values, axis=1, keepdims=True)

In [ ]:
def decode_trip(prepared, beta, emission_weight, posterior=True):
    frames, costs = prepared
    chosen = []
    posterior_rows = []
    resets = [i for i, cost in enumerate(costs) if cost is None] + [len(frames)]
    for left, right in zip(resets[:-1], resets[1:]):
        emission = [emission_weight * frames[i]["log_emission"] for i in range(left, right)]
        value = emission[0] - np.log(len(emission[0]))
        parents = [None]
        alpha = [value.copy()] if posterior else None
        for local, i in enumerate(range(left + 1, right), start=1):
            transition = transition_logprob(costs[i], beta)
            pair = value[:, None] + transition
            parents.append(pair.argmax(axis=0))
            value = emission[local] + pair.max(axis=0)
            if posterior:
                alpha.append(emission[local] + logsumexp(alpha[-1][:, None] + transition, axis=0))
        path = [int(value.argmax())]
        for local in range(len(emission) - 1, 0, -1):
            path.append(int(parents[local][path[-1]]))
        path.reverse()
        for local, i in enumerate(range(left, right)):
            chosen.append({"group_id": frames[i]["group_id"], "point_id": frames[i]["point_id"],
                           "segment_id": int(frames[i]["ids"][path[local]])})
        if posterior:
            backward = np.zeros(len(emission[-1]))
            for local in range(len(emission) - 1, -1, -1):
                i = left + local
                log_probability = alpha[local] + backward
                probability = np.exp(log_probability - logsumexp(log_probability))
                posterior_rows.append(pd.DataFrame({"group_id": frames[i]["group_id"],
                    "segment_id": frames[i]["ids"], "score": probability}))
                if local:
                    transition = transition_logprob(costs[i], beta)
                    backward = logsumexp(transition + emission[local][None, :] + backward[None, :], axis=1)
    selected = pd.DataFrame(chosen, columns=["group_id", "point_id", "segment_id"])
    probabilities = pd.concat(posterior_rows, ignore_index=True).sort_values(["group_id", "segment_id"]) if posterior_rows else None
    return selected, probabilities